# Gold · KPI Detalle por Orden

## Propósito

Mantener el máximo nivel de detalle del reporte, un registro por pedido, con historial de versiones (SCD2) para auditar la evolución de cada pedido.

## Flujo

1. En la primera carga, crear la tabla con todos los pedidos como versión vigente.
2. En las siguientes ejecuciones, comparar cada pedido contra su versión vigente.
3. Cerrar las versiones anteriores de los pedidos que cambiaron.
4. Insertar las nuevas versiones de los pedidos nuevos y de los que cambiaron.

## Destino

Capa Gold.

**Tabla:** `prod_dataco.gld_dataco.kpi_detalle_orden`

**Grano:** un registro por pedido, con historial de versiones

## Atributos que generan una nueva versión

* Estado del pedido y estado de entrega
* Ventas y utilidad
* Margen y beneficio promedio

## Control de vigencia

* `es_vigente` indica la versión actual de cada pedido.
* La pestaña Detalle Orden del Excel muestra solo la versión vigente.

## Resultado esperado

El detalle por orden actualizado, usado para decisiones puntuales y reclamos.

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog", "prod_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "gld_dataco", "Esquema Gold")

gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

GOLD_ORDER_SUMMARY_FQN = f"{gold_catalog}.{gold_schema}.order_summary"
GOLD_KPI_DETALLE_ORDEN_FQN = f"{gold_catalog}.{gold_schema}.kpi_detalle_orden"

print(f"✓ Origen: {GOLD_ORDER_SUMMARY_FQN}")
print(f"✓ Destino: {GOLD_KPI_DETALLE_ORDEN_FQN}")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control", key="id_ejecucion", default="", debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_kpi_detalle_orden"
CAPA = "gold"

spark.sql(f"""
INSERT INTO prod_dataco.metadata.etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 3: Leer versión actual desde order_summary
# ============================================================

from pyspark.sql import functions as F

df_orden_actual = spark.table(GOLD_ORDER_SUMMARY_FQN).select(
    "order_id", "market", "order_region", "customer_segment", "shipping_mode",
    "order_status", "delivery_status",
    "total_sales", "total_profit", "profit_margin_pct", "avg_benefit_per_order",
    "total_quantity", "es_entrega_a_tiempo", "shipping_variance", "es_sospechoso_fraude"
)

print(f"✓ Pedidos en order_summary: {df_orden_actual.count()}")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 4: Crear tabla si no existe (primera carga)
# ============================================================

from delta.tables import DeltaTable

if not spark.catalog.tableExists(GOLD_KPI_DETALLE_ORDEN_FQN):

    df_inicial = (
        df_orden_actual
        .withColumn("orden_detalle_key", F.monotonically_increasing_id())
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    (
        df_inicial.write.format("delta").mode("overwrite").saveAsTable(GOLD_KPI_DETALLE_ORDEN_FQN)
    )

    print(f"✓ Tabla {GOLD_KPI_DETALLE_ORDEN_FQN} creada con {df_inicial.count()} pedidos (carga inicial)")
    CARGA_INICIAL = True
else:
    print(f"✓ Tabla {GOLD_KPI_DETALLE_ORDEN_FQN} ya existe — se procesa como SCD2")
    CARGA_INICIAL = False

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 5: Lógica SCD2 (solo si la tabla ya existía)
# ------------------------------------------------------------
# Columnas que disparan una nueva versión: order_status,
# delivery_status, total_sales, total_profit, profit_margin_pct,
# avg_benefit_per_order.
# ============================================================

if not CARGA_INICIAL:

    delta_detalle_orden = DeltaTable.forName(spark, GOLD_KPI_DETALLE_ORDEN_FQN)

    df_vigente = delta_detalle_orden.toDF().filter(F.col("es_vigente") == True)

    COLUMNAS_COMPARADAS = [
        "order_status", "delivery_status",
        "total_sales", "total_profit", "profit_margin_pct", "avg_benefit_per_order"
    ]

    condicion_cambio = " OR ".join([f"v.{c} <> n.{c}" for c in COLUMNAS_COMPARADAS])

    df_comparado = (
        df_orden_actual.alias("n")
        .join(df_vigente.alias("v"), on="order_id", how="left")
        .selectExpr(
            "n.*",
            "v.orden_detalle_key AS orden_detalle_key_existente",
            f"CASE WHEN v.order_id IS NULL THEN 'NUEVO' "
            f"WHEN {condicion_cambio} THEN 'CAMBIO' "
            f"ELSE 'SIN_CAMBIO' END AS tipo_evento"
        )
    )

    df_a_cerrar = df_comparado.filter(F.col("tipo_evento") == "CAMBIO")
    df_a_insertar = df_comparado.filter(F.col("tipo_evento").isin("NUEVO", "CAMBIO"))

    print(f"✓ Pedidos nuevos: {df_comparado.filter(F.col('tipo_evento') == 'NUEVO').count()}")
    print(f"✓ Pedidos con cambios (nueva versión): {df_a_cerrar.count()}")
    print(f"✓ Pedidos sin cambios: {df_comparado.filter(F.col('tipo_evento') == 'SIN_CAMBIO').count()}")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 6: Cerrar versiones viejas (solo los que cambiaron)
# ============================================================

if not CARGA_INICIAL and df_a_cerrar.count() > 0:

    ids_con_cambio = [row["order_id"] for row in df_a_cerrar.select("order_id").collect()]

    delta_detalle_orden.update(
        condition=(F.col("order_id").isin(ids_con_cambio)) & (F.col("es_vigente") == True),
        set={"es_vigente": F.lit(False), "fecha_fin_vigencia": F.current_date()}
    )

    print(f"✓ {len(ids_con_cambio)} versiones anteriores cerradas")
elif not CARGA_INICIAL:
    print("✓ No hay versiones que cerrar")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 7: Insertar nuevas versiones (nuevos + cambiados)
# ============================================================

if not CARGA_INICIAL and df_a_insertar.count() > 0:

    max_key = spark.sql(f"SELECT MAX(orden_detalle_key) AS m FROM {GOLD_KPI_DETALLE_ORDEN_FQN}").collect()[0]["m"] or 0

    df_nuevas_versiones = (
        df_a_insertar
        .drop("orden_detalle_key_existente", "tipo_evento")
        .withColumn("orden_detalle_key", max_key + F.monotonically_increasing_id() + 1)
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    df_nuevas_versiones.write.format("delta").mode("append").saveAsTable(GOLD_KPI_DETALLE_ORDEN_FQN)

    print(f"✓ {df_nuevas_versiones.count()} nuevas versiones insertadas")
elif not CARGA_INICIAL:
    print("✓ No hay nuevas versiones que insertar")

In [0]:
# ============================================================
# GLD_KPI_DETALLE_ORDEN — Celda 8: ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE prod_dataco.metadata.etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")